In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from collections.abc import Iterable
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity
from kebab.utils.io_helpers import resolve_path

In [ ]:
# whether to save statistics as JSON files
save_statistics = False

In [ ]:
# (fragments or pairs)
dataset_path = Path.cwd().parent / "data" / "REBEL" / "linking" / "test" / "rebel_linking_dataset.jsonl"
# dataset_path = Path.cwd().parent / "data" / "REBEL" / "v1" / "clustering" / "test" / "rebel_clustering_dataset.jsonl"

# entity map
fragment_to_entity_map_path = (
    Path.cwd().parent / "data" / "REBEL" / "v1" / "linking" / "base" / "rebel_fragment_to_entity_map.jsonl"
)

fragment_to_entity_map = defaultdict(list)
with open(fragment_to_entity_map_path, "r", encoding="utf-8") as f:
    for line in f:
        fragment_id, entity_id = json.loads(line)
        fragment_to_entity_map[fragment_id] = entity_id

print(f"Loaded entity map for {len(fragment_to_entity_map)} fragments.")

Load the fragments

In [ ]:
fragments = []

# we will not be using fragments with no names
fragments_with_no_names = 0


def load_jsonl(file_path: Path) -> Iterable[ResolvedWikidataEntity]:
    """Load a jsonl file containing either ResolvedWikidataEntity objects or tuples of such objects."""
    is_tuples = None

    with open(file_path, encoding="utf-8") as f:
        for line in f:
            if is_tuples is None:
                t = json.loads(line)
                is_tuples = isinstance(t, list)

            if is_tuples:
                left, right = json.loads(line)
                yield ResolvedWikidataEntity.from_dict(left)
                yield ResolvedWikidataEntity.from_dict(right)
            else:
                yield ResolvedWikidataEntity.from_json(line)


seen = set()

for fragment in load_jsonl(dataset_path):
    if not fragment.names:
        fragments_with_no_names += 1
        del fragment
        continue

    prop_str = fragment.property_values_str()

    if prop_str in seen:
        del fragment
        continue

    seen.add(prop_str)

    if not fragment.entity_id and "fragment_id" in fragment.metadata and fragment_to_entity_map is not None:
        fragment.entity_id = fragment_to_entity_map[fragment.metadata["fragment_id"]]

    # reduce memory footprint
    if "doc_id" in fragment.metadata:
        del fragment.metadata["doc_id"]

    if "source_text_hash" in fragment.metadata:
        del fragment.metadata["source_text_hash"]

    if "fragment_id" in fragment.metadata:
        del fragment.metadata["fragment_id"]

    fragment.evidence_map = None  # type: ignore
    fragment.source_ids = None  # type: ignore

    fragments.append(fragment)

print(f"Loaded {len(fragments):,d} fragments, ignored {fragments_with_no_names:,d} fragments with no names")

Example fragment

In [ ]:
fragments[0]

In [ ]:
# compute counts of property occurrence and entity types of the fragments
property_counts = defaultdict(int)
type_counts = defaultdict(int)

for fragment in fragments:
    for prop_name, prop_value in fragment.properties.items():
        if prop_value:
            property_counts[prop_name] += 1

    if fragment.wikidata_type:
        for ent_type in fragment.wikidata_type:
            type_counts[ent_type] += 1

Top properties by occurrence in the fragments
---

In [ ]:
df = (
    pd.DataFrame(property_counts.items(), columns=["property", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

if save_statistics:
    df.to_csv("property_occurrence.csv", index=False)

df[:20]

Top entity types of the fragments
---

In [ ]:
df = (
    pd.DataFrame(type_counts.items(), columns=["type", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

if save_statistics:
    df.to_csv("type_occurrence.csv", index=False)

df[:20]

Fragments per entity
---

In [ ]:
entity_ids = [fragment.entity_id for fragment in fragments]
counts = pd.Series(entity_ids).value_counts().value_counts().sort_index()

fig = px.bar(counts, x=counts.index.astype(str), y=counts.values, title="Fragments per entity", text=counts.values)
fig.update_layout(xaxis_title="", yaxis_title="Count", width=800, height=600)
fig.show()

avg_fragments_per_entity = pd.Series(entity_ids).value_counts().mean()
print(f"Average fragments per entity: {avg_fragments_per_entity:.2f}")

Fragments sizes (if merged)
---

In [ ]:
merge_counts = [fragment.metadata["merge_count"] if "merge_count" in fragment.metadata else 1 for fragment in fragments]

counts = pd.Series(merge_counts).value_counts().sort_index()
fig = px.bar(
    counts, x=counts.index.astype(str), y=counts.values, title="Fragment sizes (if merged)", text=counts.values
)
fig.update_layout(xaxis_title="Fragment size", yaxis_title="Count", width=800, height=600)
fig.show()

Properties overlap
---

In [ ]:
# for each property how often that two distinct fragments have (1) a value for this property, and (2) the same value this property
property_value_counts = defaultdict(lambda: defaultdict(int))
entity_value_counts = defaultdict(int)

for entity_id in fragments:
    for prop_name, values in entity_id.properties.items():
        entity_value_counts[prop_name] += 1
        for value in values:
            property_value_counts[prop_name][value] += 1

entity_count = len(fragments)

rows = []
for prop_name, value_counts in property_value_counts.items():
    arr = np.array(list(value_counts.values()))
    ent_val_count = entity_value_counts[prop_name]
    ent_probs = arr / entity_count
    cond_ent_probs = arr / ent_val_count
    prob = (ent_probs**2).sum()
    cond_prob = (cond_ent_probs**2).sum()
    ent_fraction = ent_val_count / entity_count
    rows.append((prop_name, len(value_counts), prob, cond_prob, ent_fraction))

overlap_df = pd.DataFrame(
    rows, columns=["property", "distinct_value_count", "overlap_prob", "cond_overlap_prob", "entities_fraction"]
)
overlap_df = overlap_df.sort_values("overlap_prob", ascending=False)
overlap_df.head(100)